# NB12: one company file, the widened universe joined to the July Gazette features

Up to now the Gazette work has lived in its own files, keyed on CompanyNumber, and anything that
wanted both company details and distress signals had to do the join itself. The dashboard does
exactly that at build time, and so would anyone else picking this up. This notebook does the join
once and writes a single file, so a company lookup is a row lookup.

**What goes in**

The widened universe of roughly 1.5M companies across all statuses, built from the 1 August 2026
Companies House snapshot, and the 109,333 company Gazette feature table rebuilt on notices through
31 July 2026.

**What comes out**

One row per company in the universe, with the 56 Gazette columns attached where they exist and a
`has_gazette` flag saying whether they do. The row count must come out identical to the universe,
because this is a left join and companies are neither created nor lost by it.

**One thing to be honest about up front**

Only a small fraction of the universe carries any Gazette record, so most rows will have empty
feature columns. That is a finding, not a gap. The Gazette only publishes when something has gone
wrong, so silence means no insolvency event was published, which is the normal state for a healthy
company. The `has_gazette` flag exists so nobody has to guess whether a blank means "nothing
happened" or "not checked".

**The dates now line up.** An earlier version of this file paired a June company snapshot with July
notices, six weeks apart, which meant a company could read as Active while carrying a July
winding-up notice. This build uses the 1 August snapshot against notices to 31 July, so company
status is one day newer than the last notice rather than six weeks older. Anything the register
recorded about a company reflects the events we hold, not a state from before them.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

DATA = Path(r"C:\Users\visha\Lloyds_Github\data\processed")

# The two clocks in this project. They are one day apart now, which is the point
# of rebuilding on the August snapshot, but they are still two separate things and
# anything derived from this file should be stamped with whichever one applies.
CH_SNAPSHOT = "2026-08-01"         # Companies House bulk file the universe came from
NOTICE_WINDOW_END = "2026-07-31"   # what the Gazette recency features are measured against

UNIVERSE = DATA / f"filtered_bb_sme_sectors_all_status_{CH_SNAPSHOT}.csv"
FEATURES = DATA / "nb10_gazette_company_features_thru_2026-07.csv"   # 109,333 companies
MERGED_OUT = DATA / "company_master_gazette_2026-08.csv"

print("universe :", UNIVERSE.name, f"({UNIVERSE.stat().st_size / 1e6:.0f} MB)")
print("features :", FEATURES.name, f"({FEATURES.stat().st_size / 1e6:.0f} MB)")
print("out      :", MERGED_OUT.name)

universe : filtered_bb_sme_sectors_all_status_2026-08-01.csv (670 MB)
features : nb10_gazette_company_features_thru_2026-07.csv (35 MB)
out      : company_master_gazette_2026-08.csv


## Step 1: one cleaning function, applied to both sides

This is where joins in this project go wrong, so it gets its own step rather than being buried in
the merge call. Company numbers arrive in slightly different shapes on each side. The Gazette side
is pulled out of notice text by a regex that matches either eight digits or two letters followed by
six. The Companies House side comes from a bulk CSV where some column names carry a leading space
and shorter numbers are not always padded.

The function below is the same one the dashboard uses, copied rather than reimplemented so the two
cannot drift apart. Applying it to both sides means a number matches if and only if it is genuinely
the same company.

In [2]:
def clean_company_number(raw):
    'Canonical 8 character company number, or None if it cannot be one.'
    if raw is None:
        return None
    s = str(raw).strip().upper()
    if s in ("", "NAN", "NONE"):
        return None
    s = s.replace(" ", "")
    if s.isdigit():
        return s.zfill(8)
    if len(s) == 8 and s[:2].isalpha() and s[2:].isdigit():
        return s
    if len(s) <= 8:
        return s.zfill(8)
    return None


for raw, expect in [("  1234567 ", "01234567"), ("SC123456", "SC123456"),
                    ("12345678", "12345678"), ("nan", None), ("", None)]:
    got = clean_company_number(raw)
    assert got == expect, f"{raw!r} gave {got!r}, expected {expect!r}"
print("clean_company_number self-test passed")

clean_company_number self-test passed


## Step 2: load both sides

The universe is 668 MB and takes a moment. Everything is read as string so that company numbers
are never silently turned into integers, which would drop the leading zeros that half of them
depend on.

In [3]:
print("reading the widened universe (668 MB, this takes a moment) ...")
uni = pd.read_csv(UNIVERSE, dtype=str, low_memory=False)
uni.columns = [c.strip() for c in uni.columns]          # some carry a leading space
uni["cn"] = uni["CompanyNumber"].map(clean_company_number)

before = len(uni)
uni = uni[uni["cn"].notna()].drop_duplicates("cn")
print(f"universe rows: {before:,} -> {len(uni):,} "
      f"({before - len(uni):,} dropped: unusable or duplicate company number)")
print("columns:", len(uni.columns))
print()
print(uni["lifecycle"].value_counts().to_string())

reading the widened universe (668 MB, this takes a moment) ...


universe rows: 1,496,693 -> 1,496,693 (0 dropped: unusable or duplicate company number)
columns: 59

lifecycle
Trading       1380731
Fading          92265
Insolvent       23518
Distressed        179


In [4]:
ft = pd.read_csv(FEATURES, dtype=str)
ft.columns = [c.strip() for c in ft.columns]
ft["cn"] = ft["CompanyNumber"].map(clean_company_number)

before = len(ft)
ft = ft[ft["cn"].notna()].drop_duplicates("cn")
print(f"feature rows: {before:,} -> {len(ft):,}")
print("gaz columns:", len([c for c in ft.columns if c.startswith("gaz_")]))

feature rows: 109,333 -> 109,333
gaz columns: 54


## Step 3: how much of the Gazette table actually lands

Worth looking at before merging. The feature table covers every UK company with a keyed insolvency
notice, while the universe is filtered to three sectors, so most of the Gazette table falls outside
it and will not join. That is expected, but the size of the overlap is the number that decides how
much signal the dashboard has to work with, so it should be measured rather than assumed.

In [5]:
uni_keys = set(uni["cn"])
ft_keys = set(ft["cn"])
overlap = uni_keys & ft_keys

print(f"universe companies      : {len(uni_keys):,}")
print(f"Gazette feature companies: {len(ft_keys):,}")
print(f"  inside the universe    : {len(overlap):,} ({len(overlap) / len(ft_keys) * 100:.1f}% of the feature table)")
print(f"  outside it             : {len(ft_keys - uni_keys):,} (other sectors, not in our filter)")
print()
print(f"share of the universe carrying a Gazette record: {len(overlap) / len(uni_keys) * 100:.2f}%")

universe companies      : 1,496,693
Gazette feature companies: 109,333
  inside the universe    : 18,603 (17.0% of the feature table)
  outside it             : 90,730 (other sectors, not in our filter)

share of the universe carrying a Gazette record: 1.24%


## Step 4: the join

A left join from the universe. Every company keeps its row whether or not it has a Gazette record,
which is the whole point: a company with no notices is a company we checked and found nothing
against, and that is worth as much on a dashboard as a company with ten.

`company_name` is dropped off the Gazette side before merging. It came from the notice text and can
disagree with the registered name, and having two name columns invites someone to match on the
wrong one later.

In [6]:
gaz_cols = [c for c in ft.columns if c.startswith("gaz_")]
merged = uni.merge(ft[["cn"] + gaz_cols], on="cn", how="left")

merged["has_gazette"] = merged["gaz_has_any_notice"].notna().astype(int)

print("merged shape:", merged.shape)
print(f"  rows with a Gazette record   : {int(merged['has_gazette'].sum()):,}")
print(f"  rows without                 : {int((merged['has_gazette'] == 0).sum()):,}")

merged shape: (1496693, 114)
  rows with a Gazette record   : 18,603
  rows without                 : 1,478,090


## Step 5: validation

Three things have to hold. The row count must be unchanged, because a left join that changes it has
duplicated something. Company numbers must still be unique. And the number of matched rows must
equal the overlap we measured before merging, which catches the case where the join silently
matched on a differently cleaned key.

In [7]:
assert len(merged) == len(uni), f"row count changed: {len(uni):,} -> {len(merged):,}"
print("row count preserved:", f"{len(merged):,}")

assert merged["cn"].is_unique, "company numbers must remain unique"
print("company numbers unique: True")

matched = int(merged["has_gazette"].sum())
assert matched == len(overlap), f"matched {matched:,} but overlap was {len(overlap):,}"
print(f"matched rows equal the measured overlap: {matched:,}")

print("\nflagged companies by lifecycle:")
print(merged.loc[merged["has_gazette"] == 1, "lifecycle"].value_counts().to_string())

print("\nflagged companies by severity tier:")
print(merged.loc[merged["has_gazette"] == 1, "gaz_severity_tier"].value_counts().to_string())

print("\nstill Active and flagged:",
      f"{int(((merged['has_gazette'] == 1) & (merged['CompanyStatus'].str.strip() == 'Active')).sum()):,}")

row count preserved: 1,496,693


company numbers unique: True
matched rows equal the measured overlap: 18,603

flagged companies by lifecycle:
lifecycle
Insolvent     17875
Trading         592
Fading          104
Distressed       32

flagged companies by severity tier:
gaz_severity_tier
formal_insolvency    16307
terminal              1801
early_warning          469
none                    26



still Active and flagged: 592


## Step 6: save

One file, every company, Gazette features attached where they exist. The inputs are all left in
place, so this adds to the folder rather than replacing anything in it.

In [8]:
merged = merged.drop(columns=["cn"])
merged.to_csv(MERGED_OUT, index=False)

print("saved ->", MERGED_OUT.name)
print(f"  {len(merged):,} rows x {len(merged.columns)} columns")
print(f"  {MERGED_OUT.stat().st_size / 1e6:.0f} MB")

saved -> company_master_gazette_2026-08.csv
  1,496,693 rows x 113 columns
  758 MB


## What this is, and what still has to happen to it

The file is `company_master_gazette_2026-08.csv`. One row per company in the widened universe, all
the original Companies House columns, plus 54 Gazette feature columns and a `has_gazette` flag.
Joining is done, so anything downstream can read a company by number and have everything about it.

**The two clocks are now one day apart**, company register at 1 August against notices to 31 July,
where the previous build had them six weeks apart. In practice that means a company showing as
Active here really is active as of after its last notice, rather than being a stale record from
before the event. Anything stamped off this file should still carry the date of whichever side it
came from, because one day apart is not the same as identical.

**What that changes in the numbers.** Companies that entered insolvency during June and July will
have moved off Active in the register, so the count of flagged companies that are still trading
should fall compared with the June build. That is the correction working, not signal being lost.

**For the dashboard.** `dashboard/build_data.py` reads this single file plus the notice file, and
selects flagged companies with a filter on `has_gazette` rather than doing the join itself. Its
input filenames and the pinned `SNAPSHOT_DATE` both need to match whatever this notebook produced,
otherwise it rebuilds from older inputs and gives no sign that it did.